# Notebook 2: Data Preprocessing & Feature Engineering

**Site Revenue Prediction ML System - SageMaker Notebook Series**

This notebook covers:
- Feature scaling with StandardScaler
- Categorical encoding with LabelEncoder
- Boolean feature processing
- Outlier clipping (1st-99th percentile)
- Train/validation/test splits
- Saving preprocessors for inference

## SageMaker Notes
- Preprocessing should be saved as artifacts for deployment
- Use S3 for storing preprocessor pickle files
- Recommended instance: `ml.m5.xlarge` for preprocessing large datasets

In [1]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.abspath('')))

import polars as pl
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pickle
from pathlib import Path
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

PyTorch version: 2.9.1
CUDA available: False
MPS available: True


## 1. Load Data and Configuration

In [ ]:
# Configuration
DATA_PATH = '../data/processed/site_training_data.parquet'
OUTPUT_DIR = Path('./outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# Load data summary from previous notebook
try:
    with open('data_summary.json', 'r') as f:
        data_summary = json.load(f)
    print("Loaded data summary from previous notebook")
except FileNotFoundError:
    print("Data summary not found, using defaults")
    data_summary = {}

In [ ]:
# Feature definitions
NUMERIC_FEATURES = [
    "rs_Impressions", "rs_NVIs", "rs_Revenue", "rs_RevenuePerScreen",
    "avg_monthly_revenue", "log_total_revenue",
    "log_nearest_site_distance_mi", "log_min_distance_to_interstate_mi",
    "avg_household_income", "median_age", "pct_female", "pct_male",
]

CATEGORICAL_FEATURES = [
    "network", "program", "experience_type", "hardware_type", "retailer",
    "brand_fuel", "brand_restaurant", "brand_c_store", "nearest_interstate",
]

BOOLEAN_FEATURES = [
    "r_lottery_encoded", "r_government_encoded", "r_travel_and_tourism_encoded",
    "r_retail_car_wash_encoded", "r_cpg_beverage_beer_oof_encoded",
    "r_cpg_beverage_beer_vide_encoded", "r_cpg_beverage_wine_oof_encoded",
    "r_cpg_beverage_wine_video_encoded", "r_finance_credit_cards_encoded",
    "r_cpg_cbd_hemp_ingestibles_non_thc_encoded",
    "r_cpg_non_food_beverage_cannabis_medical_encoded",
    "r_cpg_non_food_beverage_cannabis_recreational_encoded",
    "r_cpg_non_food_beverage_cbd_hemp_non_thc_encoded",
    "r_alcohol_drink_responsibly_message_encoded", "r_alternative_transportation_encoded",
    "r_associations_and_npo_anti_smoking_encoded", "r_automotive_after_market_oil_encoded",
    "r_cpg_beverage_spirits_ooh_encoded", "r_cpg_beverage_spirits_video_encoded",
    "r_cpg_non_food_beverage_e_cigarette_encoded",
    "r_entertainment_casinos_and_gambling_encoded",
    "r_government_political_encoded", "r_automotive_electric_encoded",
    "r_recruitment_encoded", "r_restaurants_cdr_encoded", "r_restaurants_qsr_encoded",
    "r_retail_automotive_service_encoded", "r_retail_grocery_encoded",
    "r_retail_grocerty_with_fuel_encoded",
    "c_emv_enabled_encoded", "c_nfc_enabled_encoded", "c_open_24_hours_encoded",
    "c_sells_beer_encoded", "c_sells_diesel_fuel_encoded", "c_sells_lottery_encoded",
    "c_vistar_programmatic_enabled_encoded", "c_walk_up_enabled_encoded",
    "c_sells_wine_encoded",
    "schedule_site_encoded", "sellable_site_encoded",
]

TARGET = "avg_monthly_revenue"
TASK_TYPE = "regression"  # or "lookalike" for classification

In [ ]:
# Load data
df = pl.read_parquet(DATA_PATH)
print(f"Loaded {len(df):,} rows")

# Filter to sites with sufficient history (>11 active months)
if 'active_months' in df.columns:
    original_count = len(df)
    df = df.filter(pl.col('active_months') > 11)
    print(f"Filtered to {len(df):,} sites with >11 active months ({original_count - len(df):,} excluded)")

## 2. Create PyTorch Dataset Class

This class wraps processed tensors for efficient batch loading during training.

In [5]:
class SiteDataset(Dataset):
    """
    PyTorch Dataset for site scoring data.
    Optimized with contiguous memory layout for GPU acceleration.
    """
    
    def __init__(self, numeric_tensor, categorical_tensor, boolean_tensor, target_tensor):
        self.numeric = numeric_tensor
        self.categorical = categorical_tensor
        self.boolean = boolean_tensor
        self.target = target_tensor
    
    def __len__(self):
        return len(self.target)
    
    def __getitem__(self, idx):
        return (
            self.numeric[idx],
            self.categorical[idx],
            self.boolean[idx],
            self.target[idx],
        )

print("SiteDataset class defined")

SiteDataset class defined


## 3. Data Processor Class

This class handles all preprocessing logic: scaling, encoding, and tensor conversion.

In [6]:
class DataProcessor:
    """
    Fast data processor using Polars for CSV parsing.
    Handles encoding, scaling, and tensor conversion.
    """
    
    def __init__(self, numeric_features, categorical_features, boolean_features, 
                 target, task_type='regression'):
        self.numeric_features = numeric_features
        self.categorical_features = categorical_features
        self.boolean_features = boolean_features
        self.target = target
        self.task_type = task_type
        
        self.label_encoders = {}
        self.scaler = StandardScaler()
        self.target_scaler = StandardScaler()
        self.categorical_vocab_sizes = {}
        self._fitted = False
        
        # Track actual feature counts after processing
        self.n_numeric_features = 0
        self.n_boolean_features = 0
    
    def process(self, df):
        """Process dataframe and return tensors."""
        print(f"Processing {len(df):,} rows...")
        
        numeric_data = self._process_numeric(df)
        categorical_data = self._process_categorical(df)
        boolean_data = self._process_boolean(df)
        target_data = self._process_target(df)
        
        self._fitted = True
        
        return numeric_data, categorical_data, boolean_data, target_data
    
    def _process_numeric(self, df):
        """Extract and scale numeric features."""
        # Get available numeric columns
        available = []
        for c in self.numeric_features:
            if c in df.columns and c != self.target:
                dtype = df[c].dtype
                if dtype in [pl.Float32, pl.Float64, pl.Int8, pl.Int16, pl.Int32, pl.Int64,
                            pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64]:
                    available.append(c)
        
        print(f"Processing {len(available)} numeric features...")
        self.n_numeric_features = len(available)
        self.available_numeric = available
        
        # Process each column
        processed_cols = []
        for col in available:
            col_data = df[col].cast(pl.Float64).fill_null(0).fill_nan(0).to_numpy().astype(np.float32)
            processed_cols.append(col_data)
        
        numeric_array = np.column_stack(processed_cols) if processed_cols else np.zeros((len(df), 1), dtype=np.float32)
        
        # Clip extreme values (robust to outliers)
        for i in range(numeric_array.shape[1]):
            col_data = numeric_array[:, i]
            p1, p99 = np.percentile(col_data, [1, 99])
            numeric_array[:, i] = np.clip(col_data, p1, p99)
        
        # Fit and transform scaler
        numeric_scaled = self.scaler.fit_transform(numeric_array)
        
        # Final clip to prevent extreme scaled values
        numeric_scaled = np.clip(numeric_scaled, -10, 10)
        
        return torch.from_numpy(np.ascontiguousarray(numeric_scaled, dtype=np.float32))
    
    def _process_categorical(self, df):
        """Encode categorical features as integers for embeddings."""
        available = [c for c in self.categorical_features if c in df.columns]
        print(f"Processing {len(available)} categorical features...")
        self.available_categorical = available
        
        encoded_cols = []
        for col in available:
            col_data = df.select(col).fill_null("__MISSING__").to_series().to_list()
            
            le = LabelEncoder()
            encoded = le.fit_transform(col_data)
            self.label_encoders[col] = le
            self.categorical_vocab_sizes[col] = len(le.classes_)
            encoded_cols.append(encoded)
        
        categorical_array = np.column_stack(encoded_cols).astype(np.int64)
        return torch.from_numpy(np.ascontiguousarray(categorical_array))
    
    def _process_boolean(self, df):
        """Convert boolean features to float tensor."""
        available = [c for c in self.boolean_features if c in df.columns]
        print(f"Processing {len(available)} boolean features...")
        self.n_boolean_features = len(available) if available else 1
        self.available_boolean = available
        
        bool_cols = []
        for col in available:
            col_data = df.select(col).to_series()
            if col_data.dtype == pl.Boolean:
                values = col_data.fill_null(False).to_numpy().astype(np.float32)
            elif col_data.dtype in [pl.Int8, pl.Int16, pl.Int32, pl.Int64,
                                    pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64]:
                values = col_data.fill_null(0).to_numpy().astype(np.float32)
            else:
                values = (
                    col_data.fill_null("false")
                    .str.to_lowercase()
                    .is_in(["true", "1", "yes", "t"])
                    .to_numpy()
                    .astype(np.float32)
                )
            bool_cols.append(values)
        
        boolean_array = np.column_stack(bool_cols) if bool_cols else np.zeros((len(df), 1), dtype=np.float32)
        return torch.from_numpy(np.ascontiguousarray(boolean_array, dtype=np.float32))
    
    def _process_target(self, df):
        """Extract and process target variable."""
        print(f"Processing target: {self.target}")
        
        target_col = df[self.target]
        median_val = target_col.median()
        if median_val is None:
            median_val = 0
        
        target_data = target_col.fill_null(median_val).fill_nan(median_val).to_numpy().astype(np.float32).reshape(-1, 1)
        
        # Clip extreme values
        p1, p99 = np.percentile(target_data, [1, 99])
        target_data = np.clip(target_data, p1, p99)
        
        if self.task_type == "lookalike":
            # Classification: binarize to top 10% performers
            threshold = float(np.percentile(target_data, 90))
            self.top_performer_threshold = threshold
            binary_labels = (target_data >= threshold).astype(np.float32)
            n_positive = int(binary_labels.sum())
            print(f"Classification: threshold=${threshold:,.0f}, {n_positive}/{len(binary_labels)} positive ({n_positive/len(binary_labels)*100:.1f}%)")
            self.target_scaler = None
            return torch.from_numpy(np.ascontiguousarray(binary_labels, dtype=np.float32))
        else:
            # Regression: scale continuous target
            target_scaled = self.target_scaler.fit_transform(target_data)
            return torch.from_numpy(np.ascontiguousarray(target_scaled, dtype=np.float32))
    
    def save(self, path):
        """Save fitted preprocessors."""
        with open(path, 'wb') as f:
            pickle.dump({
                'label_encoders': self.label_encoders,
                'scaler': self.scaler,
                'target_scaler': self.target_scaler,
                'categorical_vocab_sizes': self.categorical_vocab_sizes,
                'n_numeric_features': self.n_numeric_features,
                'n_boolean_features': self.n_boolean_features,
                'available_numeric': getattr(self, 'available_numeric', []),
                'available_categorical': getattr(self, 'available_categorical', []),
                'available_boolean': getattr(self, 'available_boolean', []),
            }, f)
        print(f"Preprocessor saved to {path}")
    
    def load(self, path):
        """Load fitted preprocessors."""
        with open(path, 'rb') as f:
            data = pickle.load(f)
            self.label_encoders = data['label_encoders']
            self.scaler = data['scaler']
            self.target_scaler = data['target_scaler']
            self.categorical_vocab_sizes = data['categorical_vocab_sizes']
            self.n_numeric_features = data.get('n_numeric_features', 0)
            self.n_boolean_features = data.get('n_boolean_features', 0)
            self._fitted = True
        print(f"Preprocessor loaded from {path}")

print("DataProcessor class defined")

DataProcessor class defined


## 4. Process Data

In [7]:
# Remove target from numeric features to prevent leakage
numeric_features_no_target = [f for f in NUMERIC_FEATURES if f != TARGET]

# Initialize processor
processor = DataProcessor(
    numeric_features=numeric_features_no_target,
    categorical_features=CATEGORICAL_FEATURES,
    boolean_features=BOOLEAN_FEATURES,
    target=TARGET,
    task_type=TASK_TYPE,
)

# Process data
numeric, categorical, boolean, target = processor.process(df)

print(f"\nProcessed tensors:")
print(f"  Numeric: {numeric.shape}")
print(f"  Categorical: {categorical.shape}")
print(f"  Boolean: {boolean.shape}")
print(f"  Target: {target.shape}")

Processing 22,754 rows...
Processing 11 numeric features...
Processing 9 categorical features...
Processing 40 boolean features...
Processing target: avg_monthly_revenue

Processed tensors:
  Numeric: torch.Size([22754, 11])
  Categorical: torch.Size([22754, 9])
  Boolean: torch.Size([22754, 40])
  Target: torch.Size([22754, 1])


In [8]:
# Display vocabulary sizes for categorical features
print("\nCategorical Vocabulary Sizes:")
print("=" * 40)
for col, vocab_size in processor.categorical_vocab_sizes.items():
    print(f"  {col:30} {vocab_size:5} unique values")


Categorical Vocabulary Sizes:
  network                            4 unique values
  program                           13 unique values
  experience_type                    3 unique values
  hardware_type                     17 unique values
  retailer                        9736 unique values
  brand_fuel                       139 unique values
  brand_restaurant                 168 unique values
  brand_c_store                   8139 unique values
  nearest_interstate               227 unique values


## 5. Create Train/Validation/Test Splits

In [9]:
# Split ratios
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Get dataset size
n_samples = len(target)
print(f"Total samples: {n_samples:,}")

# Create random indices for splitting (reproducible)
torch.manual_seed(42)
indices = torch.randperm(n_samples)

# Calculate split sizes
n_train = int(n_samples * TRAIN_RATIO)
n_val = int(n_samples * VAL_RATIO)

train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]

print(f"Train: {len(train_idx):,} ({len(train_idx)/n_samples*100:.1f}%)")
print(f"Val:   {len(val_idx):,} ({len(val_idx)/n_samples*100:.1f}%)")
print(f"Test:  {len(test_idx):,} ({len(test_idx)/n_samples*100:.1f}%)")

Total samples: 22,754
Train: 15,927 (70.0%)
Val:   3,413 (15.0%)
Test:  3,414 (15.0%)


In [10]:
# Create datasets
train_dataset = SiteDataset(
    numeric[train_idx],
    categorical[train_idx],
    boolean[train_idx],
    target[train_idx],
)

val_dataset = SiteDataset(
    numeric[val_idx],
    categorical[val_idx],
    boolean[val_idx],
    target[val_idx],
)

test_dataset = SiteDataset(
    numeric[test_idx],
    categorical[test_idx],
    boolean[test_idx],
    target[test_idx],
)

print(f"\nDataset sizes:")
print(f"  Train: {len(train_dataset):,}")
print(f"  Val:   {len(val_dataset):,}")
print(f"  Test:  {len(test_dataset):,}")


Dataset sizes:
  Train: 15,927
  Val:   3,413
  Test:  3,414


## 6. Create DataLoaders

In [11]:
# DataLoader configuration
BATCH_SIZE = 4096  # Large batches work well on GPU

# Detect device and set num_workers
# NOTE: On macOS, multiprocessing with DataLoader can crash (especially Python 3.13+)
# Set num_workers=0 to use single-process data loading on macOS
import platform

if torch.cuda.is_available():
    device = 'cuda'
    pin_memory = True
    NUM_WORKERS = 4  # Use multiprocessing on Linux/CUDA
elif torch.backends.mps.is_available():
    device = 'mps'
    pin_memory = False  # MPS works best without pin_memory
    NUM_WORKERS = 0  # Disable multiprocessing on macOS (avoids fork issues)
else:
    device = 'cpu'
    pin_memory = False
    # On macOS CPU, also disable multiprocessing
    NUM_WORKERS = 0 if platform.system() == 'Darwin' else 4

print(f"Using device: {device}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Num workers: {NUM_WORKERS}")

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    drop_last=True,  # For stable batch norm
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

print(f"\nDataLoader batches:")
print(f"  Train: {len(train_loader)} batches")
print(f"  Val:   {len(val_loader)} batches")
print(f"  Test:  {len(test_loader)} batches")

Using device: mps
Batch size: 4096
Num workers: 0

DataLoader batches:
  Train: 3 batches
  Val:   1 batches
  Test:  1 batches


## 7. Verify Data Loading

In [12]:
# Get a sample batch
sample_numeric, sample_cat, sample_bool, sample_target = next(iter(train_loader))

print("Sample batch shapes:")
print(f"  Numeric:     {sample_numeric.shape}")
print(f"  Categorical: {sample_cat.shape}")
print(f"  Boolean:     {sample_bool.shape}")
print(f"  Target:      {sample_target.shape}")

print(f"\nData types:")
print(f"  Numeric:     {sample_numeric.dtype}")
print(f"  Categorical: {sample_cat.dtype}")
print(f"  Boolean:     {sample_bool.dtype}")
print(f"  Target:      {sample_target.dtype}")

Sample batch shapes:
  Numeric:     torch.Size([4096, 11])
  Categorical: torch.Size([4096, 9])
  Boolean:     torch.Size([4096, 40])
  Target:      torch.Size([4096, 1])

Data types:
  Numeric:     torch.float32
  Categorical: torch.int64
  Boolean:     torch.float32
  Target:      torch.float32


In [13]:
# Verify numeric feature scaling
print("\nNumeric feature statistics (should be ~0 mean, ~1 std after scaling):")
print(f"  Mean: {sample_numeric.mean():.4f}")
print(f"  Std:  {sample_numeric.std():.4f}")
print(f"  Min:  {sample_numeric.min():.4f}")
print(f"  Max:  {sample_numeric.max():.4f}")


Numeric feature statistics (should be ~0 mean, ~1 std after scaling):
  Mean: 0.0006
  Std:  1.0058
  Min:  -4.7523
  Max:  4.6795


## 8. Save Preprocessor and Data Artifacts

In [14]:
# Save preprocessor
processor.save(OUTPUT_DIR / 'preprocessor.pkl')

# Save processed tensors for faster loading in training notebook
torch.save({
    'numeric': numeric,
    'categorical': categorical,
    'boolean': boolean,
    'target': target,
    'train_idx': train_idx,
    'val_idx': val_idx,
    'test_idx': test_idx,
}, OUTPUT_DIR / 'processed_data.pt')

print(f"Saved processed data to {OUTPUT_DIR / 'processed_data.pt'}")

Preprocessor saved to outputs/preprocessor.pkl
Saved processed data to outputs/processed_data.pt


In [15]:
# Save configuration for model building
model_config = {
    'n_numeric': processor.n_numeric_features,
    'n_boolean': processor.n_boolean_features,
    'n_categorical': len(processor.categorical_vocab_sizes),
    'categorical_vocab_sizes': processor.categorical_vocab_sizes,
    'task_type': TASK_TYPE,
    'target': TARGET,
    'device': device,
    'batch_size': BATCH_SIZE,
}

with open(OUTPUT_DIR / 'model_config.json', 'w') as f:
    # Convert vocab_sizes to regular dict for JSON
    config_serializable = model_config.copy()
    config_serializable['categorical_vocab_sizes'] = dict(model_config['categorical_vocab_sizes'])
    json.dump(config_serializable, f, indent=2)

print(f"Saved model config to {OUTPUT_DIR / 'model_config.json'}")
print("\nModel configuration:")
for key, value in model_config.items():
    if key != 'categorical_vocab_sizes':
        print(f"  {key}: {value}")

Saved model config to outputs/model_config.json

Model configuration:
  n_numeric: 11
  n_boolean: 40
  n_categorical: 9
  task_type: regression
  target: avg_monthly_revenue
  device: mps
  batch_size: 4096


## 9. SageMaker S3 Upload (Optional)

Uncomment the cell below to upload preprocessor and data to S3 for SageMaker training.

In [ ]:
# Uncomment for SageMaker S3 upload
# import sagemaker
# from sagemaker.s3 import S3Uploader

# session = sagemaker.Session()
# bucket = session.default_bucket()
# prefix = 'site-scoring'

# # Upload preprocessor
# preprocessor_s3 = S3Uploader.upload(
#     str(OUTPUT_DIR / 'preprocessor.pkl'),
#     f's3://{bucket}/{prefix}/preprocessor'
# )
# print(f"Preprocessor uploaded to: {preprocessor_s3}")

# # Upload processed data
# data_s3 = S3Uploader.upload(
#     str(OUTPUT_DIR / 'processed_data.pt'),
#     f's3://{bucket}/{prefix}/processed'
# )
# print(f"Processed data uploaded to: {data_s3}")

---

## Summary

This notebook has:
1. ✅ Defined PyTorch Dataset class for batched data loading
2. ✅ Created DataProcessor class for feature preprocessing
3. ✅ Scaled numeric features with StandardScaler
4. ✅ Encoded categorical features with LabelEncoder
5. ✅ Processed boolean features to float tensors
6. ✅ Created train/val/test splits (70/15/15)
7. ✅ Created DataLoaders optimized for GPU training
8. ✅ Saved preprocessor for inference

**Artifacts created:**
- `outputs/preprocessor.pkl` - Fitted scalers and encoders
- `outputs/processed_data.pt` - PyTorch tensors ready for training
- `outputs/model_config.json` - Configuration for model building

**Next:** Proceed to `03_model_training.ipynb` to build and train the neural network.